# 앙상블 — 배깅 · 랜덤 포레스트 (분류 · 회귀)

수업 중 작성한 필기 `앙상블분류실습` · `앙상불회귀실습01` 을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `머신러닝` p.91~101 — 13. 앙상블)

- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.
- 그림 · **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 미리 만들어 두었고, 아래 코드를 실행하면 같은 그림이 다시 그려집니다.
- 단일 결정 트리 회귀와 불안정성은 [`tree_regression.ipynb`](./tree_regression.ipynb) 에 정리했습니다.

> 실습 데이터 `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` (70,692행) · `house_price.csv` (506행) 는 이 노트북과 **같은 폴더**에 있어야 합니다.

In [ ]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 앙상블 학습 (교재 p.92~93)

**여러 개의 단일 결정 트리 모델을 연결해서 더 강력한 모델을 만드는 기법.** 악기 하나보다 밴드가 낫다.

| 종류 | 방식 |
|------|------|
| **Bagging** (배깅) | 데이터를 무작위로 여러 번 뽑아 트리를 **따로따로(병렬)** 학습 → 결과를 모은다 · **오늘** |
| **Boosting** (부스팅) | 앞 트리가 틀린 것을 다음 트리가 **이어서(순차)** 보완 |
| **Stacking** (스태킹) | 여러 모델의 예측을 다시 **입력으로 쓰는** 모델을 하나 더 학습 |

### 결과를 모으는 방법 — 투표 / 평균

| 문제 | 방법 | 내용 |
|------|------|------|
| 분류 | **직접 투표** (Hard Voting) | 각 모델이 낸 **클래스 레이블**의 다수결 |
| 분류 | **간접 · 소프트 투표** (Soft Voting) | 각 모델이 낸 **클래스별 확률의 평균** → 가장 높은 클래스. 확신도까지 반영해서 보통 더 좋다 |
| 회귀 | **평균** | 연속값은 다수결이 안 되므로 각 모델 예측값의 **평균** |

In [ ]:
import numpy as np

# 트리 3개가 한 샘플에 대해 낸 [클래스0 확률, 클래스1 확률]
proba = np.array([
    [0.45, 0.55],   # 트리 1 : 1 (아슬아슬)
    [0.40, 0.60],   # 트리 2 : 1 (아슬아슬)
    [0.95, 0.05],   # 트리 3 : 0 (아주 확신)
])

hard = np.bincount(proba.argmax(axis=1)).argmax()     # 레이블 다수결
soft = proba.mean(axis=0).argmax()                    # 확률 평균 → 최댓값
print("각 트리의 예측 :", proba.argmax(axis=1))
print("직접 투표(Hard) :", hard, "← 1 이 2표")
print("확률 평균        :", proba.mean(axis=0).round(3))
print("소프트 투표(Soft):", soft, "← 트리 3 의 강한 확신이 반영됨")

## 2. 배깅 · 페이스팅 · 랜덤 패치 · 랜덤 서브스페이스 (교재 p.94~95)

### 행(샘플)을 어떻게 뽑나

| 방식 | 추출 | `bootstrap` |
|------|------|-------------|
| **배깅** (Bootstrap Aggregating) | 학습용에서 **복원 추출** — 중복 허용 | `True` |
| **페이스팅** (Pasting) | 학습용에서 **비복원 추출** — 중복 없음 | `False` |

각 서브 데이터로 모델을 따로 학습한 뒤 결과를 모은다 (분류 : 다수결, 회귀 : 평균).

![부트스트랩 복원 추출 시뮬레이션](./images/anim_04_bootstrap_sampling.gif)

In [ ]:
# 부트스트랩 직접 해 보기 : 0 ~ 9 번 데이터 10개에서 10번 '복원 추출'
rng = np.random.RandomState(7)
for i in range(1, 4):
    sample = np.sort(rng.randint(0, 10, size=10))         # 같은 번호가 또 나올 수 있다
    missing = sorted(set(range(10)) - set(sample))
    print(f"서브 데이터 {i} : {sample}  → 빠진 번호 {missing}")

# 데이터가 많으면 한 번도 안 뽑히는 비율은 약 37% (= (1 - 1/n)^n → 1/e)
n = 10000
picked = np.unique(np.random.RandomState(0).randint(0, n, size=n)).size
print(f"\n{n}개에서 {n}번 복원 추출 → 뽑힌 고유 데이터 {picked / n:.1%}, 한 번도 안 뽑힘 {1 - picked / n:.1%}")

> 트리마다 **조금씩 다른 데이터**로 학습하니 트리끼리 서로 다른 실수를 하고, 다수결 · 평균을 내면 그 실수가 **상쇄**된다.

### 열(특성)까지 뽑기 — 다채로움(Diversity) 확보

| 기법 | 샘플(행) | 특성(열) | 매개변수 |
|------|---------|---------|---------|
| **랜덤 패치** (Random Patches) | 무작위 | 무작위 | `bootstrap=True` & `max_features < 1.0` |
| **랜덤 서브스페이스** (Random Subspaces) | **전체** 사용 | 무작위 | `bootstrap=False` & `max_features < 1.0` |

→ 고차원 데이터에서 일부 **강한 특성에 모델이 편향**되는 것을 막는다.

## 3. 랜덤 포레스트 (교재 p.96~97)

**결정 트리를 기본 모델로 쓰는, 랜덤 패치 기반의 대표적인 배깅 앙상블.**

- 트리마다 부트스트랩 샘플로 학습
- **트리의 무작위성** — 노드를 나눌 때 전체 특성을 다 보지 않고, 무작위로 고른 일부(분류는 기본 $\sqrt{n\_features}$ 개) 중에서 최선의 분할을 고른다
- 결과 : 분류는 **투표**(scikit-learn 은 확률 평균 = 소프트 투표), 회귀는 **평균**

```python
from sklearn.ensemble import RandomForestClassifier      # 분류
rf = RandomForestClassifier(random_state=0)             # n_estimators=100 (트리 100개) 가 기본값

from sklearn.ensemble import RandomForestRegressor       # 회귀
rf_reg = RandomForestRegressor(random_state=0)
```

### 🎬 숲이 자라는 과정 — 트리가 늘수록 경계가 매끄러워진다

초승달 두 개가 맞물린 데이터(`make_moons`)에서 트리를 1 → 100 개로 늘리며 **확률 평균(소프트 투표)** 의 경계를 그렸다.

![랜덤 포레스트가 자라는 과정](./images/anim_05_forest_growing.gif)

- 트리 1개는 계단 모양 경계에 정확도 76.7% (트리 하나하나의 평균은 85.0%)
- 트리가 늘수록 경계가 곡선을 따라 매끄러워지고, 100개에서 **91.3%**
- 색이 진할수록 트리들이 **한목소리**로 투표한 곳, 흐릴수록 의견이 갈린 곳이다

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 곡선 경계 데이터 500개 → 학습 70 : 평가 30
P, lab = make_moons(n_samples=500, noise=0.3, random_state=0)
Pa, Pb, la, lb = train_test_split(P, lab, test_size=0.3, random_state=0, stratify=lab)

# 트리 100개짜리 숲을 한 번만 학습하고, 앞에서부터 k 개 트리만 골라 '작은 숲' 을 흉내 낸다
forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(Pa, la)

xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 200), np.linspace(-1.3, 1.8, 200))
grid_pts = np.c_[xx.ravel(), yy.ravel()]
tree_proba = np.array([t.predict_proba(grid_pts)[:, 1] for t in forest.estimators_])   # 트리별 확률
test_proba = np.array([t.predict_proba(Pb)[:, 1] for t in forest.estimators_])

n_list = [1, 2, 3, 5, 10, 20, 50, 100]
fig, ax = plt.subplots(figsize=(6, 4.5))

def draw(i):
    k = n_list[i]
    prob = tree_proba[:k].mean(axis=0).reshape(xx.shape)      # 소프트 투표 = 확률 평균
    acc = accuracy_score(lb, (test_proba[:k].mean(axis=0) >= 0.5).astype(int))
    ax.clear()
    ax.contourf(xx, yy, prob, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.4)
    ax.contour(xx, yy, prob, levels=[0.5], colors='k')       # 확률 0.5 = 결정 경계
    ax.scatter(Pa[:, 0], Pa[:, 1], c=la, cmap='bwr', s=8)
    ax.set_title(f'트리 {k}개 → 평가용 정확도 {acc:.1%}')

anim = FuncAnimation(fig, draw, frames=len(n_list), interval=1000)
plt.close(fig)
HTML(anim.to_jshtml())

## 4. 랜덤 포레스트 분류 — 당뇨병 여부 (교재 p.98~99)

Day 023 에서 단일 결정 트리로 분류했던 그 데이터 · 그 분할이다. (트리 : 제한 없음 65.5%, `max_depth=7` 74.1%)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 불러오기 (노트북과 같은 폴더)
df = pd.read_csv('./diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
print("모양 :", df.shape)

# 특성(21개) / 정답
X_data = df.drop(columns=['Diabetes_binary'])
y_data = df['Diabetes_binary']

# 80 : 20, 정답 비율(50 : 50) 유지
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=0, stratify=y_data
)
print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)
print("학습용 정답 비율 :", y_train.value_counts(normalize=True).round(3).tolist())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 모델 생성 : 트리 100개(기본값), 무작위성 고정
#   n_jobs=-1 : CPU 코어를 모두 써서 트리를 동시에 학습 (결과는 같고 빨라진다)
rf = RandomForestClassifier(random_state=0, n_jobs=-1)

# 학습 : 트리 100개가 각자 부트스트랩 샘플로 학습
rf.fit(X_train, y_train)
print("학습 완료! 트리 개수 :", len(rf.estimators_))

# 예측 : 평가용 특성(X_test)만 넣는다 — 정답은 주지 않는다
pred_test_rf = rf.predict(X_test)

# 평가 : accuracy_score(정답, 예측값)
acc_rf = accuracy_score(y_test, pred_test_rf)
print(f"평가용 정확도 : {acc_rf:.4f}")
print(f"학습용 정확도 : {accuracy_score(y_train, rf.predict(X_train)):.4f}")

> 정확도 **0.7402**. 그런데 학습용은 **99.5%** — 트리 하나하나는 끝까지 자라서 학습용을 외운 상태다.
> 그래도 단일 트리(65.5%)보다 **8.5%p** 높다. 아래에서 그 이유를 확인한다.

### 트리 100개 각각 vs 숲 전체

In [ ]:
# 숲 안의 트리 100개를 하나씩 꺼내 정확도 측정
#   (각 트리는 학습할 때 특성 이름 없이 배열로 받았으므로 .values 로 넣는다)
tree_acc = [accuracy_score(y_test, tree.predict(X_test.values)) for tree in rf.estimators_]

print(f"트리 하나하나 : 평균 {np.mean(tree_acc):.4f}  (최저 {min(tree_acc):.4f} ~ 최고 {max(tree_acc):.4f})")
print(f"100개 투표    : {acc_rf:.4f}")

In [ ]:
# 트리 개수(n_estimators) 를 늘려 가며 정확도 변화 (약 10~20초)
n_list = [1, 2, 5, 10, 20, 50, 100, 200]
acc_list = []
for n in n_list:
    model = RandomForestClassifier(n_estimators=n, random_state=0, n_jobs=-1).fit(X_train, y_train)
    acc_list.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(n_list, acc_list, marker='o', label='랜덤 포레스트')
plt.axhline(np.mean(tree_acc), color='gray', linestyle='--', label='트리 하나 평균')
plt.xscale('log')
plt.xticks(n_list, n_list)
plt.xlabel('n_estimators (트리 개수)')
plt.ylabel('정확도')
plt.legend()
plt.show()

print(pd.Series(acc_list, index=n_list, name='정확도').round(4))

![트리 개수에 따른 정확도](./images/07_rf_n_estimators.png)

> - 혼자서는 평균 **65.9%** 인 트리들이, 모여서 투표하면 **74.0%** 가 된다 — 앙상블의 힘
> - 트리를 늘릴수록 오르다가 **100개 근처에서 포화** (200개도 74.0%). 트리를 더 늘리면 시간만 더 든다

### 규제 매개변수 + 특성 중요도

In [ ]:
# 트리 하나하나의 깊이를 제한하면? (교재 p.84 규제 매개변수는 랜덤 포레스트에도 그대로 쓴다)
for params in [{}, {'max_depth': 10}, {'min_samples_leaf': 20}]:
    model = RandomForestClassifier(random_state=0, n_jobs=-1, **params).fit(X_train, y_train)
    print(f"{str(params):24s} → 학습용 {accuracy_score(y_train, model.predict(X_train)):.4f} / "
          f"평가용 {accuracy_score(y_test, model.predict(X_test)):.4f}")

In [ ]:
# 특성 중요도 : 트리 100개에서 평균 낸 값 (합계 1)
importance = pd.Series(rf.feature_importances_, index=X_data.columns).sort_values()

plt.figure(figsize=(8, 5))
importance.tail(10).plot.barh()
plt.xlabel('특성 중요도')
plt.show()

print(importance.sort_values(ascending=False).head(6).round(3))

![랜덤 포레스트 특성 중요도](./images/08_rf_feature_importance.png)

> - `max_depth=10` 으로 트리를 제한하자 평가용 **75.3%** 로 더 올랐다 (학습용 77.4% 로 차이도 줄었다)
> - 단일 트리(Day 023)는 `HighBP` 하나에 중요도 0.45 가 몰렸는데, 랜덤 포레스트는 **BMI 0.17 · Age 0.13 · GenHlth 0.11 …** 로 고르게 나뉜다.
>   노드마다 특성을 무작위로 골라야 해서 **강한 특성 하나에 편향되지 않기** 때문이다 (교재 p.95 랜덤 패치의 장점)

> **📝 고친 부분** (`앙상블분류실습`)
> - `print("평가용 데이터:", y_train.index)` → `y_train` 은 **학습용 정답**. X · y 짝 확인은 `==` 로 한 번에
> - 주석 "평가용 데이터를 이용한 예측 — **정답을 줘서** 예측 값" → `predict()` 에는 **문제(X_test)만** 넣는다. 정답을 주면 시험이 아니다
> - `print("예측 값:", acc_rf)` → 출력하는 건 예측값이 아니라 **정확도**
> - 주석 "1. 정답과 2. **평가용 데이터**" → `accuracy_score(정답, **예측값**)`
> - 계산만 하고 쓰지 않던 `ratio_train` → 비율 출력으로 사용
> - 주석 "클래스 호출할 때 항상 (대문자)" → 파이썬 관례상 **클래스 이름은 대문자로 시작**(`RandomForestClassifier`)하고, 그 클래스로 만든 **객체 변수는 소문자**(`rf`)로 쓴다
> - 경로 `C:\Users\Jinho\work\…` → `./`, `warnings.filterwarnings('ignore')` 제거, `print(df)` → 모양 출력
> - 트리 하나 vs 숲, 트리 개수 · 규제 매개변수 비교, 특성 중요도 추가

## 5. 랜덤 포레스트 회귀 — 보스턴 집값 (교재 p.100~101)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error as rmse

df_house = pd.read_csv('./house_price.csv')
X_h = df_house.drop(columns=['medv'])
y_h = df_house['medv']

# 75 : 25 — random_state 를 고정해야 결과를 다시 만들 수 있다
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.25, random_state=0
)

# 모델 생성 · 학습 · 예측
rf_reg = RandomForestRegressor(random_state=0, n_jobs=-1)
rf_reg.fit(X_train_h, y_train_h)
pred_test_rf = rf_reg.predict(X_test_h)

# 평가 : 회귀는 정확도가 아니라 RMSE
rmse_rf = rmse(y_test_h, pred_test_rf)
print(f"랜덤 포레스트 RMSE : {rmse_rf:.2f}")

In [ ]:
# 같은 분할에서 모델 비교 (Day 021 · 023 결과 포함)
from sklearn.linear_model import LinearRegression

dt_reg = DecisionTreeRegressor(random_state=0).fit(X_train_h, y_train_h)
lr = LinearRegression().fit(X_train_h, y_train_h)

compare = pd.Series({
    'LinearRegression (Day 021)': rmse(y_test_h, lr.predict(X_test_h)),
    'SVR C=100 (Day 023, 표준화)': 4.19,
    'DecisionTreeRegressor': rmse(y_test_h, dt_reg.predict(X_test_h)),
    'RandomForestRegressor': rmse_rf,
}, name='평가용 RMSE').round(2).sort_values()
print(compare)

In [ ]:
# 정답 vs 예측 산점도 : 대각선(정답 = 예측)에 가까울수록 좋다
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, (name, pred) in zip(axes, [('결정 트리', dt_reg.predict(X_test_h)), ('랜덤 포레스트', pred_test_rf)]):
    ax.scatter(y_test_h, pred, s=15)
    ax.plot([0, 50], [0, 50], 'k:')
    ax.set_title(f'{name}  RMSE {rmse(y_test_h, pred):.2f}')
    ax.set_xlabel('실제 집값')
axes[0].set_ylabel('예측 집값')
plt.show()

print(pd.Series(rf_reg.feature_importances_, index=X_h.columns).sort_values(ascending=False).head(4).round(3))

![결정 트리 vs 랜덤 포레스트](./images/04_boston_dt_vs_rf.png)

> **결과 해석**
> - 랜덤 포레스트 RMSE **4.07** — 단일 트리(5.41) · 선형 회귀(5.46)보다 좋고, 하이퍼파라미터를 고른 SVR(4.19)보다도 낫다 (표준화도 필요 없다)
> - 단일 트리는 대각선에서 멀리 튄 점이 많고, 랜덤 포레스트는 대각선 근처에 **모여 있다**
> - 중요한 특성은 방 개수 `rm` 0.42 · 저소득층 비율 `lstat` 0.41

> **📝 고친 부분** (`앙상불회귀실습01`)
> - `train_test_split` 에 **`random_state` 가 없어서** 실행할 때마다 데이터가 다르게 나뉘었다 → 필기의 RMSE **2.85** 는 다시 만들 수 없는 값이다. `random_state=0` 으로 고정 (Day 021 · 023 과 같은 분할이라 비교 가능)
> - 회귀 문제에 `accuracy_score` 임포트 → 정확도는 **분류** 지표. 회귀는 RMSE 만
> - `print("예상 값:", rmse_rf)` → 출력하는 건 **RMSE**(평균적으로 틀린 크기)
> - 학습용 데이터 전체를 `print(X_train)` 하던 것 → 모양만 출력
> - 파일 이름 `앙상**불**` → 앙상**블**

## 6. 오늘 정리 (앙상블)

| 항목 | 내용 |
|------|------|
| 앙상블 | 여러 트리를 묶어 더 강한 모델 — 배깅 · 부스팅 · 스태킹 |
| 모으기 | 분류 : 직접 투표 / **소프트 투표**(확률 평균), 회귀 : **평균** |
| 배깅 | **복원 추출**(부트스트랩)로 서브 데이터 → 트리마다 따로 학습 |
| 페이스팅 | **비복원 추출** |
| 랜덤 패치 / 서브스페이스 | 행 + 열 무작위 / 열만 무작위 → 강한 특성 편향 방지 |
| 랜덤 포레스트 | 결정 트리 + 랜덤 패치 배깅, 노드마다 특성 일부만 검토 |

| 데이터 | 단일 결정 트리 | 랜덤 포레스트 |
|--------|---------------|---------------|
| 당뇨병 (정확도 ↑) | 65.5% | **74.0%** (`max_depth=10` 이면 75.3%) |
| 보스턴 (RMSE ↓) | 5.41 | **4.07** |